In [ ]:
from pathlib import Path
from Physical_Quantity.Physical_Data_Record import SuperCorrRecord
from Physical_Quantity.Super_Correlation_Process import SuperCorrProcessor

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-----------------########### UltraDMRG计算的三个键方向的超导关联函数<Δ_ij^†·Δ_kl>数据提取与作图 ################----------------------
### 基本参数
N = 200
a = 1.0

### ================= 文件夹路径变量（需要修改的部分）=====================
δ = "d1_10"               # 空穴率
version = "50_54"         # reference bond
bond_dim = "1500"         # bond dimension
t_J = "t_J_3"             # t/J

base_dir = Path(f'Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_super_corr/super_corr_{version}')

### 将三个键方向A、B、C的数据提取并封装成SuperCorrRecord对象列表
records = {}
for bond in 'ABC':
    for channel in 'abcd':
        key = f'scs{bond}{channel}'
        fname = f'{key}_fix_refbond_{version}.json'
        records[key] = SuperCorrRecord.extract_data(
            (base_dir / fname).read_text(encoding='utf-8'), N, a
        )

### 方向显示名映射字典
bond_labels = {
    'A': "A key (blue bond)",
    'B': "B key (green bond)",
    'C': "C key (orange bond)",
}

### 检测超导关联对称性并组合超导关联函数
super_corr_dict_UltraDMRG = {}
for bond, label in bond_labels.items():
    ## 取出当前 bond 对应的四个 SuperCorrRecord 对象
    recs = [records[f'scs{bond}{channel}'] for channel in 'abcd']

    ## 对称性检查
    err_ad, err_bc = SuperCorrProcessor.symmetry_check(*recs)
    print(f"{bond}键方向的通道a与通道d的相对误差为：{err_ad}；通道b与通道c的相对误差为：{err_bc}")

    ## 组合超导关联函数，存入最终字典
    super_corr_dict_UltraDMRG[label] = SuperCorrProcessor.Combined_super_correlation(*recs)

### 可视化
SuperCorrProcessor.Super_Corr_Visualization_three(super_corr_dict_UltraDMRG)